# Working with Remote LLM APIs: Gemini (Solutions)

In this lab, you will call the Google Gemini API from Python. The goal is to see the actual API request: the URL, headers, JSON payload, response JSON, and parsed text.

We use short synthetic examples so the lab stays focused on API mechanics rather than data cleaning.


## How to Use This Notebook

Run the setup cells first. Put a temporary Gemini key in `GEMINI_API_KEY` in Setup 1, then work through the questions in order.

The cells that call the remote API print `REMOTE API CALLED: YES`. Repeated calls pause briefly to avoid Gemini free-tier rate limits.


## Setup: Minimal Gemini API Code

The setup is intentionally short. It defines the key, model, synthetic data, prompt builders, and two small functions used later.


### Setup 1: Imports, Key, Model, and Constants

Paste a temporary class key into `GEMINI_API_KEY`. `GEMINI_PAUSE_SECONDS` spaces out repeated calls for the free-tier rate limit.


In [1]:
# Setup 1
import json
import time

import requests

GEMINI_API_KEY = ""  # paste a temporary class key here
GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/" + GEMINI_MODEL + ":generateContent"
GEMINI_PAUSE_SECONDS = 15

ALLOWED_LABELS = ["access_barrier", "housing_stress", "health_need", "other"]


### Setup 2: Short Synthetic Examples and Prompts

These toy examples are deliberately short so the API request stays easy to inspect.


In [2]:
# Setup 2
CLASSIFICATION_LIMIT = 6
SUMMARY_LIMIT = 2
EXTRACTION_LIMIT = 2
REDACTION_LIMIT = 2

toy_notes = [
    {"id": "N001", "text": "Bus cancelled; missed clinic.", "human_label": "access_barrier"},
    {"id": "N002", "text": "Rent arrears and possible eviction.", "human_label": "housing_stress"},
    {"id": "N003", "text": "Medication side effects; needs follow-up.", "human_label": "health_need"},
]

applied_notes = toy_notes + [
    {"id": "A004", "text": "Park lighting is better; no service problem reported.", "human_label": "other"},
    {"id": "A005", "text": "Care worker could not enter the block because the intercom was broken.", "human_label": "access_barrier"},
    {"id": "A006", "text": "Temporary accommodation has damp and the family needs housing advice.", "human_label": "housing_stress"},
    {"id": "A007", "text": "Resident reports panic attacks and asks for a mental health appointment.", "human_label": "health_need"},
]

public_complaint_extracts = [
    {"id": "C001", "text": "Booking is hard. Phone line busy. Some residents have no internet."},
    {"id": "C002", "text": "Residents at a sheltered block say the lift is often broken. Wheelchair users wait downstairs. Repair updates are unclear."},
]
public_complaint_extract = public_complaint_extracts[0]["text"]

deidentified_service_notes = [
    {"id": "E001", "text": "Area=Riverside. Issue=missed clinic after bus change. Support=transport advice."},
    {"id": "E002", "text": "Area=Hilltop. Issue=food bank referral after benefit delay. Support=voucher and welfare advice."},
    {"id": "E003", "text": "Area=Eastgate. Issue=unsafe stairs in temporary housing. Support=repairs escalation."},
]
deidentified_service_note = deidentified_service_notes[0]["text"]

sensitive_training_examples = [
    {"id": "R001", "text": "Maria Jones, SW1A 1AA, 07123 456789, missed clinic after bus cancellation."},
    {"id": "R002", "text": "Ahmed Khan, E2 6AA, 07911 222333, asked for rent arrears advice after reduced hours."},
    {"id": "R003", "text": "Leah Smith, M14 5PT, 07700 900123, asked for help after mould worsened child asthma."},
]
sensitive_training_example = sensitive_training_examples[0]["text"]

def make_label_prompt(text):
    return "Return exactly one label and nothing else: access_barrier, housing_stress, health_need, or other. Do not use Markdown.\nText: " + text

def make_summary_prompt(text):
    return "Start directly with two bullet points and then one uncertainty sentence. Do not write an introduction.\n- First concrete issue: ...\n- Second concrete issue: ...\nUncertainty: ...\nText: " + text

def make_extraction_prompt(text):
    return "Return only JSON with area, main_issue, requested_support.\nNote: " + text

def redact_for_api(text):
    for name in ["Maria Jones", "Ahmed Khan", "Leah Smith"]:
        text = text.replace(name, "[NAME]")
    for postcode in ["SW1A 1AA", "E2 6AA", "M14 5PT"]:
        text = text.replace(postcode, "[POSTCODE]")
    for phone in ["07123 456789", "07911 222333", "07700 900123"]:
        text = text.replace(phone, "[PHONE]")
    return text


### Setup 3: Repeat Later Calls

Question 5 writes out the first Gemini request step by step. Later questions use `ask_gemini(...)` so the notebook does not repeat the same HTTP code every time.


In [3]:
# Setup 3
def ask_gemini(prompt, max_output_tokens=200, temperature=None):
    print("Waiting", GEMINI_PAUSE_SECONDS, "seconds for Gemini rate limits...")
    time.sleep(GEMINI_PAUSE_SECONDS)

    headers = {"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"}
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": max_output_tokens},
    }
    if temperature is not None:
        payload["generationConfig"]["temperature"] = temperature

    start = time.time()
    response = requests.post(GEMINI_URL, headers=headers, json=payload, timeout=120)
    if response.status_code == 429:
        print("Rate limit hit. Waiting once more, then retrying.")
        time.sleep(GEMINI_PAUSE_SECONDS)
        start = time.time()
        response = requests.post(GEMINI_URL, headers=headers, json=payload, timeout=120)
    if response.status_code != 200:
        raise RuntimeError(response.text)

    api_json = response.json()
    candidate = api_json["candidates"][0]
    if "parts" not in candidate["content"]:
        print(json.dumps(api_json, indent=2))
        raise RuntimeError("Gemini returned no text. Increase max_output_tokens and rerun.")
    api_text = candidate["content"]["parts"][0]["text"].strip()
    return {
        "model": GEMINI_MODEL,
        "status": response.status_code,
        "elapsed_seconds": time.time() - start,
        "payload": payload,
        "json": api_json,
        "text": api_text,
    }


### Setup 4: Label Cleanup

This small function cleans labels for later comparison.


In [4]:
# Setup 4
def normalise_label(text):
    cleaned = text.strip().lower().replace("-", "_").replace(" ", "_")
    for label in ALLOWED_LABELS:
        if label in cleaned:
            return label
    return "invalid"

print("Setup loaded: Gemini", GEMINI_MODEL, "with", len(applied_notes), "synthetic examples")


Setup loaded: Gemini gemini-2.5-flash with 7 synthetic examples


## Part A: Gemini Setup and Prompt (Approx. 20 minutes)

In this part, you check that the notebook can see one valid Gemini setup and build the first prompt.

Keep the task small. A short classification prompt is easier to debug than a long, complex request. The same prompt will later be sent to whichever Gemini you configured.


### Question 1: Check Gemini Setup

Confirm that the notebook has a key string and the Gemini model name. If `Key pasted` is `False`, paste a key into Setup 1 and rerun the setup cells.


In [5]:
# Answer 1
print("Key pasted:", GEMINI_API_KEY != "")
print("Model:", GEMINI_MODEL)
print("URL:", GEMINI_URL)


Key pasted: True
Model: gemini-2.5-flash
URL: https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent


### Question 2: Build a Toy Prompt

Start with one synthetic note. Your aim is to build a prompt that is easy to inspect before sending.

Programming steps:

- choose the first note;
- pass the note text into `make_label_prompt(...)`;
- print the prompt and check the allowed labels are clear.


In [6]:
# Answer 2
selected_note = toy_notes[0]
prompt = make_label_prompt(selected_note["text"])

print("Selected note:", selected_note["id"])
print(prompt)


Selected note: N001
Return exactly one label and nothing else: access_barrier, housing_stress, health_need, or other. Do not use Markdown.
Text: Bus cancelled; missed clinic.


## Part B: Build and Inspect One Gemini Request (Approx. 25 minutes)

Do not send a request until you understand what is in it.

In this part, you build the Gemini JSON payload and inspect the URL, headers, prompt, and generation settings.


### Question 3: Build the Gemini Request Payload

The payload is the JSON body that will be sent to Gemini.

Programming steps:

- create a `payload` dictionary;
- use enough `maxOutputTokens` for Gemini to answer;
- print the payload with indentation;
- check the prompt and output limit.


In [7]:
# Answer 3
payload = {
    "contents": [{"parts": [{"text": prompt}]}],
    "generationConfig": {"maxOutputTokens": 200},
}

print(json.dumps(payload, indent=2))


{
  "contents": [
    {
      "parts": [
        {
          "text": "Return exactly one label and nothing else: access_barrier, housing_stress, health_need, or other. Do not use Markdown.\nText: Bus cancelled; missed clinic."
        }
      ]
    }
  ],
  "generationConfig": {
    "maxOutputTokens": 200
  }
}


### Question 4: Inspect the Redacted Request

The API key belongs in the request header. Redact it before printing or sharing a request.

Programming steps:

- create the request headers;
- make a redacted copy of the headers;
- print the URL, redacted headers, and payload.


In [8]:
# Answer 4
headers = {"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"}
redacted_headers = headers.copy()
redacted_headers["x-goog-api-key"] = "[REDACTED]"

print("URL:", GEMINI_URL)
print("Headers:")
print(json.dumps(redacted_headers, indent=2))
print("Payload:")
print(json.dumps(payload, indent=2))


URL: https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent
Headers:
{
  "x-goog-api-key": "[REDACTED]",
  "Content-Type": "application/json"
}
Payload:
{
  "contents": [
    {
      "parts": [
        {
          "text": "Return exactly one label and nothing else: access_barrier, housing_stress, health_need, or other. Do not use Markdown.\nText: Bus cancelled; missed clinic."
        }
      ]
    }
  ],
  "generationConfig": {
    "maxOutputTokens": 200
  }
}


## Part C: Make, Parse, and Compare Gemini Calls (Approx. 35 minutes)

These cells make real API calls. They are intentionally small: one prompt, short output, synthetic text.

If a call fails, use the traceback and the status-code guidance later in the lab to identify the problem. Common causes include missing key, wrong model name, quota limits, unsupported parameters, and network timeouts.


### Question 5: Make One API Call

This is the first live remote model call. It is written out so you can see the request, response, JSON parsing, and text extraction.

Programming steps:

- send `payload` to `GEMINI_URL` with `requests.post(...)`;
- if the free-tier rate limit is hit, wait and retry once;
- stop if Gemini returns another error;
- parse the JSON response;
- extract the response text;
- store the main results in `api_result`;
- print one clear confirmation line beginning `REMOTE API CALLED: YES`.


In [9]:
# Answer 5
start = time.time()
response = requests.post(GEMINI_URL, headers=headers, json=payload, timeout=120)
elapsed_seconds = time.time() - start

if response.status_code == 429:
    print("Rate limit hit. Waiting once, then retrying.")
    time.sleep(GEMINI_PAUSE_SECONDS)
    start = time.time()
    response = requests.post(GEMINI_URL, headers=headers, json=payload, timeout=120)
    elapsed_seconds = time.time() - start

if response.status_code != 200:
    raise RuntimeError(response.text)

api_json = response.json()
candidate = api_json["candidates"][0]
if "parts" not in candidate["content"]:
    print(json.dumps(api_json, indent=2))
    raise RuntimeError("Gemini returned no text. Increase maxOutputTokens and rerun.")

api_text = candidate["content"]["parts"][0]["text"].strip()
api_result = {
    "model": GEMINI_MODEL,
    "status": response.status_code,
    "elapsed_seconds": elapsed_seconds,
    "payload": payload,
    "json": api_json,
    "text": api_text,
}

print(f"REMOTE API CALLED: YES | task=single_note_classification | model={api_result['model']} | status={api_result['status']} | elapsed_seconds={api_result['elapsed_seconds']:.2f}")
print("Text:", api_result["text"])


REMOTE API CALLED: YES | task=single_note_classification | model=gemini-2.5-flash | status=200 | elapsed_seconds=1.72
Text: health_need


### Question 6: Parse Response Text

The parser should work on the real response from Question 5.

Programming steps:

- read the response text from Gemini's JSON;
- normalise the text into a label;
- print both values.


In [10]:
# Answer 6
parsed_text = api_result["json"]["candidates"][0]["content"]["parts"][0]["text"].strip()
parsed_label = normalise_label(parsed_text)
print("Parsed text:", parsed_text)
print("Normalised label:", parsed_label)


Parsed text: health_need
Normalised label: health_need


### Question 7: Compare Two Parameter Settings

Parameters are part of the method. To reduce quota use, reuse the default result from Question 5 and make one new low-temperature call.

Programming steps:

- store the Question 5 result as the default setting;
- call Gemini once with low temperature;
- store the output and normalised label for both settings.


In [11]:
# Answer 7
parameter_results = [
    {
        "name": "default_from_question_5",
        "text": api_result["text"],
        "label": normalise_label(api_result["text"]),
    }
]

low_result = ask_gemini(prompt, max_output_tokens=200, temperature=0.0)
parameter_results.append({
    "name": "low_temperature",
    "text": low_result["text"],
    "label": normalise_label(low_result["text"]),
})

print("REMOTE API CALLED: YES", parameter_results[-1])
print(parameter_results)


Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES {'name': 'low_temperature', 'text': 'access_barrier', 'label': 'access_barrier'}
[{'name': 'default_from_question_5', 'text': 'health_need', 'label': 'health_need'}, {'name': 'low_temperature', 'text': 'access_barrier', 'label': 'access_barrier'}]


## Part D: Logging, Failure Handling, and Validation (Approx. 25 minutes)

Remote calls are research steps. Record enough information to know what model, prompt, and settings produced the answer.


### Question 8: Create a Reproducibility Log Entry

Create a compact log entry for the first real API call.

Programming steps:

- build a small log dictionary;
- include Gemini, model, prompt hash, output limit, and status;
- print the log entry as JSON.


In [12]:
# Answer 8
log_entry = {
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "task": "single_note_classification",
    "model": api_result["model"],
    "prompt": prompt,
    "max_output_tokens": 200,
    "status": api_result["status"],
}
print(json.dumps(log_entry, indent=2))


{
  "time": "2026-07-01 12:46:16",
  "task": "single_note_classification",
  "model": "gemini-2.5-flash",
  "prompt": "Return exactly one label and nothing else: access_barrier, housing_stress, health_need, or other. Do not use Markdown.\nText: Bus cancelled; missed clinic.",
  "max_output_tokens": 200,
  "status": 200
}


### Question 9: Explain API Failure Codes

When a real call fails, the status code helps identify the next fix.

This question does not call the API. It prints a small reference table for common status codes.


In [13]:
# Answer 9
status_messages = {
    200: "Success",
    400: "Bad request",
    401: "Authentication problem",
    403: "Permission or billing problem",
    429: "Rate or quota limit",
    500: "Server problem",
}
for status_code, message in status_messages.items():
    print(status_code, "->", message)


200 -> Success
400 -> Bad request
401 -> Authentication problem
403 -> Permission or billing problem
429 -> Rate or quota limit
500 -> Server problem


### Question 10: Validate Against the Human Label

The model output is not evidence until it is checked.

Programming steps:

- compare the model label with the human label;
- store the comparison in a dictionary;
- print the result.


In [14]:
# Answer 10
validation_result = {
    "note_id": selected_note["id"],
    "model": api_result["model"],
    "human_label": selected_note["human_label"],
    "model_text": api_result["text"],
    "model_label": normalise_label(api_result["text"]),
}
validation_result["match"] = validation_result["model_label"] == validation_result["human_label"]

print(json.dumps(validation_result, indent=2))


{
  "note_id": "N001",
  "model": "gemini-2.5-flash",
  "human_label": "access_barrier",
  "model_text": "health_need",
  "model_label": "health_need",
  "match": false
}


## Part E: Applied Health and Social Workflows (Approx. 45 minutes)

Now apply the same real API pattern to four small tasks:

- batch classification;
- summarisation;
- structured extraction;
- redaction before sending text.

The examples remain synthetic and small, but the programming pattern is the same pattern used in larger workflows. Gemini switching should not change the workflow: build prompt, call model, parse response, validate or inspect.


### Programming Pattern for the Applied Tasks

Each applied task follows the same pattern: build a prompt, call Gemini, inspect the output, and decide whether the output is usable. The notebook keeps the number of live calls small because free-tier Gemini projects may allow only a few requests per minute.


### Question 11: Batch Classify Applied Notes

Batch work means repeating the same operation over several examples. The setup cell includes extra social examples; `CLASSIFICATION_LIMIT` controls how many live calls this cell makes.

Programming steps:

- loop over the first `CLASSIFICATION_LIMIT` applied notes;
- build one label prompt per note;
- call Gemini once per note;
- normalise the output;
- print a confirmation line for each real API call;
- compare with the human label.


In [15]:
# Answer 11
batch_rows = []

for note in applied_notes[:CLASSIFICATION_LIMIT]:
    note_prompt = make_label_prompt(note["text"])
    result = ask_gemini(note_prompt, max_output_tokens=200)
    model_label = normalise_label(result["text"])
    row = {
        "id": note["id"],
        "human_label": note["human_label"],
        "model_text": result["text"],
        "model_label": model_label,
        "match": model_label == note["human_label"],
    }
    batch_rows.append(row)
    print("REMOTE API CALLED: YES", row)


Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES {'id': 'N001', 'human_label': 'access_barrier', 'model_text': 'health_need', 'model_label': 'health_need', 'match': False}
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES {'id': 'N002', 'human_label': 'housing_stress', 'model_text': 'housing_stress', 'model_label': 'housing_stress', 'match': True}
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES {'id': 'N003', 'human_label': 'health_need', 'model_text': 'health_need', 'model_label': 'health_need', 'match': True}
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES {'id': 'A004', 'human_label': 'other', 'model_text': 'other', 'model_label': 'other', 'match': True}
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES {'id': 'A005', 'human_label': 'access_barrier', 'model_text': 'access_barrier', 'model_label': 'access_barrier', 'match': True}
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLE

### Question 12: Evaluate the Batch Classifier

A classifier should be evaluated as a classifier, not just inspected row by row.

Use the `batch_rows` object from Question 11 to calculate overall accuracy and count invalid outputs.

This is still a tiny synthetic evaluation set, so the numbers are not research evidence. The point is to practise the evaluation structure.


In [16]:
# Answer 12
total_rows = len(batch_rows)
correct_rows = 0
invalid_rows = 0

for row in batch_rows:
    if row["match"]:
        correct_rows = correct_rows + 1
    if row["model_label"] not in ALLOWED_LABELS:
        invalid_rows = invalid_rows + 1

accuracy = correct_rows / total_rows
classifier_evaluation = {
    "n": total_rows,
    "accuracy": accuracy,
    "invalid_outputs": invalid_rows,
}

print("Accuracy:", correct_rows, "/", total_rows, "=", round(accuracy, 3))
print("Invalid outputs:", invalid_rows)


Accuracy: 5 / 6 = 0.833
Invalid outputs: 0


### Question 13: Summarise Public Complaint Extracts

Summarisation is different from classification. The output is prose, so validation requires reading for accuracy and uncertainty. `SUMMARY_LIMIT` controls how many complaint extracts this cell sends.

Programming steps:

- loop over the first `SUMMARY_LIMIT` complaint extracts;
- build a summary prompt that forbids an introductory sentence;
- call Gemini with a larger output limit;
- print each summary.


In [17]:
# Answer 13
summary_results = []

for complaint in public_complaint_extracts[:SUMMARY_LIMIT]:
    summary_prompt = make_summary_prompt(complaint["text"])
    summary_result = ask_gemini(summary_prompt, max_output_tokens=600)
    row = {"id": complaint["id"], "summary": summary_result["text"]}
    summary_results.append(row)
    print(f"REMOTE API CALLED: YES | task=summarisation | id={complaint['id']} | model={summary_result['model']} | status={summary_result['status']} | elapsed_seconds={summary_result['elapsed_seconds']:.2f}")
    print(row["summary"])

summary_text = summary_results[0]["summary"]


Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES | task=summarisation | id=C001 | model=gemini-2.5-flash | status=200 | elapsed_seconds=4.51
- First concrete issue: Phone line busy.
- Second concrete issue: Some residents have no internet.
Uncertain
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES | task=summarisation | id=C002 | model=gemini-2.5-flash | status=200 | elapsed_seconds=1.96
- First concrete issue: The lift in the sheltered block is frequently out of order.
- Second concrete issue: Wheelchair users are unable to leave their homes or access their flats when the lift is broken, forcing them to wait downstairs.
Uncertainty: Information regarding repair status and timelines for the lift is not being clearly communicated to residents.


### Question 14: Extract Structured Fields

Structured extraction is useful because the output can be parsed into a Python dictionary. `EXTRACTION_LIMIT` controls how many de-identified service notes this cell sends.

Programming steps:

- loop over the first `EXTRACTION_LIMIT` de-identified notes;
- ask for valid JSON only;
- call Gemini;
- parse the returned text with `json.loads`;
- inspect the keys.

If parsing fails, the prompt or model output needs attention. This is a real methodological issue: a model can produce plausible text that is not valid machine-readable JSON.


In [18]:
# Answer 14
extracted_rows = []

for service_note in deidentified_service_notes[:EXTRACTION_LIMIT]:
    extraction_prompt = make_extraction_prompt(service_note["text"])
    extraction_result = ask_gemini(extraction_prompt, max_output_tokens=300)
    print(f"REMOTE API CALLED: YES | task=structured_extraction | id={service_note['id']} | model={extraction_result['model']} | status={extraction_result['status']} | elapsed_seconds={extraction_result['elapsed_seconds']:.2f}")
    clean_json_text = extraction_result["text"].replace("```json", "").replace("```", "").strip()
    extracted_fields = json.loads(clean_json_text)
    extracted_fields["id"] = service_note["id"]
    extracted_rows.append(extracted_fields)
    print(json.dumps(extracted_fields, indent=2))


Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES | task=structured_extraction | id=E001 | model=gemini-2.5-flash | status=200 | elapsed_seconds=1.27
{
  "area": "Riverside",
  "main_issue": "missed clinic after bus change",
  "requested_support": "transport advice",
  "id": "E001"
}
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES | task=structured_extraction | id=E002 | model=gemini-2.5-flash | status=200 | elapsed_seconds=1.12
{
  "area": "Hilltop",
  "main_issue": "food bank referral after benefit delay",
  "requested_support": "voucher and welfare advice",
  "id": "E002"
}


### Question 15: Minimise and Redact Before Sending Text

Before sending text to a hosted model, ask whether each detail is needed. `REDACTION_LIMIT` controls how many sensitive synthetic examples this cell sends after redaction.

Programming steps:

- loop over the first `REDACTION_LIMIT` sensitive examples;
- redact obvious identifiers from each synthetic note;
- build a classification prompt from the redacted note;
- call Gemini;
- print the redacted note and model label.

This is not a complete anonymisation procedure. It is a programming demonstration of data minimisation before a remote API call.


In [19]:
# Answer 15
redaction_rows = []

for example in sensitive_training_examples[:REDACTION_LIMIT]:
    redacted_note = redact_for_api(example["text"])
    redacted_prompt = make_label_prompt(redacted_note)
    redacted_result = ask_gemini(redacted_prompt, max_output_tokens=200)
    row = {
        "id": example["id"],
        "redacted_note": redacted_note,
        "model_text": redacted_result["text"],
        "model_label": normalise_label(redacted_result["text"]),
    }
    redaction_rows.append(row)
    print(f"REMOTE API CALLED: YES | task=redacted_note_classification | id={example['id']} | model={redacted_result['model']} | status={redacted_result['status']} | elapsed_seconds={redacted_result['elapsed_seconds']:.2f}")
    print(json.dumps(row, indent=2))


Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES | task=redacted_note_classification | id=R001 | model=gemini-2.5-flash | status=200 | elapsed_seconds=1.73
{
  "id": "R001",
  "redacted_note": "[NAME], [POSTCODE], [PHONE], missed clinic after bus cancellation.",
  "model_text": "access_barrier",
  "model_label": "access_barrier"
}
Waiting 15 seconds for Gemini rate limits...
REMOTE API CALLED: YES | task=redacted_note_classification | id=R002 | model=gemini-2.5-flash | status=200 | elapsed_seconds=1.39
{
  "id": "R002",
  "redacted_note": "[NAME], [POSTCODE], [PHONE], asked for rent arrears advice after reduced hours.",
  "model_text": "housing_stress",
  "model_label": "housing_stress"
}


### Question 16: Final Mini-Case

Finish by documenting one responsible remote-model workflow.

Your report should include:

- the task;
- the Gemini and model;
- privacy step;
- parameters;
- validation evidence;
- likely failure modes.


In [20]:
# Answer 16
mini_case = {
    "task": "classify short synthetic comments",
    "model": api_result["model"],
    "privacy_step": "use synthetic or redacted text",
    "validation": validation_result,
    "batch_accuracy": classifier_evaluation["accuracy"],
    "summaries_run": len(summary_results),
    "extractions_run": len(extracted_rows),
    "redactions_run": len(redaction_rows),
    "possible_failure": "missing key, quota limit, wrong model, or unexpected output",
}

print(json.dumps(mini_case, indent=2))


{
  "task": "classify short synthetic comments",
  "model": "gemini-2.5-flash",
  "privacy_step": "use synthetic or redacted text",
  "validation": {
    "note_id": "N001",
    "model": "gemini-2.5-flash",
    "human_label": "access_barrier",
    "model_text": "health_need",
    "model_label": "health_need",
    "match": false
  },
  "batch_accuracy": 0.8333333333333334,
  "summaries_run": 2,
  "extractions_run": 2,
  "redactions_run": 2,
  "possible_failure": "missing key, quota limit, wrong model, or unexpected output"
}


## End of Lab Checklist

By the end of this lab, you should have:

- pasted or checked a Gemini API key;
- built a Gemini request payload;
- inspected a redacted request;
- made real Gemini API calls without exceeding the free-tier rate limit;
- parsed response JSON into text;
- compared parameter settings;
- logged model and prompt information;
- validated model output against a human label;
- batch-classified short synthetic notes;
- summarised, extracted fields, and redacted a note before sending it.
